# Play MASA Safety Gridworlds

Set `ENV_NAME` to one of `"island_navigation"`, `"conveyor_belt"`, or `"sokoban"`. The preview cell renders inline with `rgb_array`, and the last cell opens a `pygame` window for keyboard play.

In [ ]:
ENV_NAME = "sokoban"
SEED = 0

In [ ]:
from PIL import Image

from masa.envs.discrete.conveyor_belt import ConveyorBelt
from masa.envs.discrete.island_navigation import IslandNavigation
from masa.envs.discrete.sokoban import Sokoban

ENV_BUILDERS = {
    "island_navigation": IslandNavigation,
    "conveyor_belt": ConveyorBelt,
    "sokoban": Sokoban,
}

print("Available:", sorted(ENV_BUILDERS))
preview_env = ENV_BUILDERS[ENV_NAME](render_mode="rgb_array")
obs, info = preview_env.reset(seed=SEED)
print("reset info:", info)
Image.fromarray(preview_env.render())

In [ ]:
import pygame

ACTION_KEYS = {
    pygame.K_RIGHT: 0,
    pygame.K_UP: 1,
    pygame.K_LEFT: 2,
    pygame.K_DOWN: 3,
    pygame.K_d: 0,
    pygame.K_w: 1,
    pygame.K_a: 2,
    pygame.K_s: 3,
}

def play_env(env_name: str, seed: int = 0):
    env = ENV_BUILDERS[env_name](render_mode="human")
    obs, info = env.reset(seed=seed)
    finished = False
    print("Controls: arrows or WASD move, R resets, Q or Escape quits.")
    print("reset info:", info)

    try:
        while True:
            action = None
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    return
                if event.type != pygame.KEYDOWN:
                    continue
                if event.key in (pygame.K_q, pygame.K_ESCAPE):
                    return
                if event.key == pygame.K_r:
                    obs, info = env.reset(seed=seed)
                    finished = False
                    print("reset info:", info)
                    continue
                if event.key in ACTION_KEYS and not finished:
                    action = ACTION_KEYS[event.key]

            if action is not None:
                obs, reward, terminated, truncated, info = env.step(action)
                finished = terminated or truncated
                print({
                    "reward": reward,
                    "terminated": terminated,
                    "truncated": truncated,
                    "info": info,
                })

            pygame.time.wait(16)
    finally:
        env.close()

play_env(ENV_NAME, seed=SEED)